# Eczema v2 - FINISH (after the training was resumed from the terminal)

Use this **only** because Step C of `PrecisionSkin_Eczema_v2_Pipeline.ipynb` was interrupted and training was finished from the terminal.
It does what the end of Step C does: **test evaluation -> ONNX export -> JSON**. It does not train anything. Kernel: `.venv_det`.

Run the two cells in order. Then go back to `PrecisionSkin_Eczema_v2_Pipeline.ipynb`, run **Step 0** (settings cell) and then **Step D** (the comparison).
Do not run Step A, B or C again.

## 1 - Is the training really finished?

In [1]:
from pathlib import Path
import pandas as pd

WORK = Path.home() / "PrecisionSkinV1" / "detector_work"
RUN = WORK / "runs" / "detector_v2_yolov8n_imgsz640_seed42"
res = pd.read_csv(RUN / "results.csv"); res.columns = [c.strip() for c in res.columns]
print("epochs finished:", int(res["epoch"].max()), "of 100 (early stopping may end sooner)")
best_i = res["metrics/mAP50-95(B)"].idxmax()
print("best epoch:", int(res.loc[best_i, "epoch"]), "| val mAP50 %.3f | val mAP50-95 %.3f" % (res.loc[best_i, "metrics/mAP50(B)"], res.loc[best_i, "metrics/mAP50-95(B)"]))
print("best.pt exists:", (RUN / "weights" / "best.pt").exists(), "| last.pt exists:", (RUN / "weights" / "last.pt").exists())
import subprocess
running = subprocess.run("ps aux | grep -E 'yolo|ultralytics' | grep -v grep | wc -l", shell=True, capture_output=True, text=True).stdout.strip()
print("training processes still running:", running, "  (should be 0 before you continue; if not 0, wait)")

epochs finished: 100 of 100 (early stopping may end sooner)
best epoch: 55 | val mAP50 0.308 | val mAP50-95 0.110
best.pt exists: True | last.pt exists: True
training processes still running: 0   (should be 0 before you continue; if not 0, wait)


## 2 - Test evaluation + ONNX export  (a few minutes)

In [2]:
import json, time, shutil, hashlib, datetime
import numpy as np
from ultralytics import YOLO
import onnxruntime as ort

MODEL, IMGSZ, DATASET_VERSION, CONF, SEEDS = "yolov8n.pt", 640, "v2", 0.25, [42]
CLASSES = ["acne", "hyperpigmentation", "eczema"]
ds_dir = WORK / "dataset_v2"
DATA_YAML = ds_dir / "data.yaml"
BEST = RUN / "weights" / "best.pt"
assert BEST.exists() and DATA_YAML.exists()
man = pd.read_csv(ds_dir / "dataset_manifest.csv")

# ---- test evaluation (same code as Stage 7 of the export notebook) ----
met = YOLO(str(BEST)).val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, conf=0.001, plots=True)
per = {}
for k, ci in enumerate(met.box.ap_class_index):
    p, r, ap50, ap = met.box.class_result(k)
    per[CLASSES[int(ci)]] = {"precision": float(p), "recall": float(r), "mAP50": float(ap50), "mAP50_95": float(ap)}
results = {42: {"overall": {"precision": float(met.box.mp), "recall": float(met.box.mr), "mAP50": float(met.box.map50), "mAP50_95": float(met.box.map)}, "per_class": per}}
print(pd.DataFrame([dict(cls=c, **v) for c, v in per.items()]).round(3).to_string(index=False))
test_n = man.query("split == 'test'").groupby("condition").size()
print("test photos per class:", test_n.to_dict())

# ---- ONNX export + CPU latency (same code as Stage 8) ----
onnx_src = Path(YOLO(str(BEST)).export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True, dynamic=False))
EXP = WORK / "export"; EXP.mkdir(parents=True, exist_ok=True)
onnx_path = EXP / ("lesion_detector_%s_%s_imgsz%d.onnx" % (DATASET_VERSION, Path(MODEL).stem, IMGSZ))
shutil.copy2(onnx_src, onnx_path)
so = ort.SessionOptions(); so.intra_op_num_threads = 4
sess = ort.InferenceSession(str(onnx_path), so, providers=["CPUExecutionProvider"])
inp = sess.get_inputs()[0]
x = np.random.rand(1, 3, IMGSZ, IMGSZ).astype("float32")
for _ in range(3):
    sess.run(None, {inp.name: x})
ts = []
for _ in range(20):
    t0 = time.perf_counter(); out = sess.run(None, {inp.name: x}); ts.append((time.perf_counter() - t0) * 1000)
lat = {"median_ms": round(float(np.median(ts)), 1), "p95_ms": round(float(np.percentile(ts, 95)), 1), "threads": 4}
print("ONNX ok | output", out[0].shape, "| CPU latency:", lat)

meta = {
    "model": "PrecisionSkin lesion detector", "weights": MODEL, "dataset_version": DATASET_VERSION,
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
    "classes": CLASSES, "imgsz": IMGSZ, "default_conf": CONF, "nms_iou": 0.45,
    "onnx_sha256": hashlib.sha256(onnx_path.read_bytes()).hexdigest(),
    "test_metrics_by_seed": {str(k): v for k, v in results.items()},
    "split_images": man.groupby("split").size().to_dict(),
    "test_photos_per_class": test_n.to_dict(),
    "box_origin_counts": man.groupby("box_origin").size().to_dict(),
    "cpu_latency": lat,
    "preprocess": "letterbox to IMGSZxIMGSZ (pad 114), RGB, float32 / 255, NCHW",
    "output": "[1, 4+num_classes, N]: rows cx,cy,w,h (pixels of the letterboxed input) then class scores; no objectness; apply NMS",
    "note": "Research prototype. Boxes show candidate lesion locations; not a diagnosis. Eczema is a pilot class. Training was resumed from last.pt (epoch 77) after a memory stall.",
}
json.dump(meta, open(EXP / (onnx_path.stem + ".json"), "w"), indent=2, default=str)
print("saved:", onnx_path); print("saved:", EXP / (onnx_path.stem + ".json"))
print("\nNEXT: open PrecisionSkin_Eczema_v2_Pipeline.ipynb -> run Step 0, then Step D.")

Ultralytics 8.4.172 🚀 Python-3.13.15 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Model summary (fused): 72 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 184.3±61.8 MB/s, size: 72.4 KB)
val: Scanning /home/bautista/PrecisionSkinV1/detector_work/dataset_v2/test/labels... 566 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 566/566 1.2Kit/s 0.5s<0.1s
val: New cache created: /home/bautista/PrecisionSkinV1/detector_work/dataset_v2/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 5.8it/s 6.3s0.1s
                   all        566       3563      0.358      0.387      0.303      0.107
                  acne        370       3118      0.435      0.398      0.358      0.113
     hyperpigmentation        164        380      0.405      0.379      0.312      0.125
                eczema         32         65      0.235      0.38